# M2A4 — Exemplos práticos de GPU e aceleração

Este notebook mantém **os códigos exatamente como apresentados nos slides**.  
O objetivo das explicações abaixo é preparar a leitura de cada célula, destacando **o que será executado**, **qual conceito arquitetural está sendo ilustrado** e **como interpretar o resultado**.


## Célula 1 — Loop Python vs NumPy vetorizado

Nesta célula, comparamos duas formas de multiplicar vetores grandes:

1. **Loop Python puro**: cada multiplicação é executada de forma sequencial, com overhead do interpretador a cada iteração.
2. **Operação vetorizada do NumPy**: a expressão `a * b` delega o trabalho para código otimizado em baixo nível, explorando melhor o hardware da CPU.

### O que observar
- O resultado numérico é o mesmo nas duas abordagens.
- O tempo de execução tende a ser muito menor no NumPy.
- Isso acontece porque o NumPy reduz o custo do interpretador e aproveita bibliotecas otimizadas, frequentemente com uso de vetorização e melhor acesso à memória.

### Conceito didático
Este exemplo mostra que, antes mesmo de ir para GPU, **a forma de expressar o problema já altera drasticamente o desempenho**.


In [11]:
import numpy as np
import time

n = 10_000_000
a = np.random.rand(n)
b = np.random.rand(n)

# Abordagem 1: Loop Python (interpretado, sequencial)
start = time.time()
c = [a[i] * b[i] for i in range(n)]
print(f"Loop Python:  {time.time() - start:.3f}s")

# Abordagem 2: Vetorizado NumPy (SIMD em C, paralelo)
start = time.time()
c = a * b
print(f"NumPy vetor:  {time.time() - start:.3f}s")

Loop Python:  1.316s
NumPy vetor:  0.089s


## Célula 2 — Numba JIT: mantendo o loop, mas compilando para código nativo

Nesta célula, o código continua com estrutura de **loop explícito**, mas agora usamos `@njit` do Numba.

### O que acontece
- Na **primeira chamada**, o Numba compila a função para código nativo otimizado.
- Na **segunda chamada**, a função já está compilada, então medimos apenas a execução.

### O que observar
- O estilo do código continua parecido com Python tradicional.
- O desempenho melhora porque o loop deixa de ser interpretado linha por linha.
- Esse tipo de abordagem é importante quando queremos manter uma lógica imperativa, mas sem pagar todo o custo do Python puro.

### Conceito didático
Aqui o aluno percebe que há um caminho intermediário entre:
- escrever tudo em Python puro; e
- reescrever a solução inteira em outra tecnologia.


In [12]:
from numba import njit
import numpy as np
import time

@njit
def mult_loop(a, b):
    c = np.empty(len(a))
    for i in range(len(a)):
        c[i] = a[i] * b[i]
    return c

n = 10_000_000
a = np.random.rand(n)
b = np.random.rand(n)
print (a[:10])
# 1ª chamada: inclui compilação JIT
_ = mult_loop(a, b)

# 2ª chamada: já compilado
start = time.time()
c = mult_loop(a, b)
print(f"Numba JIT: {time.time()-start:.4f}s")

[0.93596486 0.75126429 0.44995792 0.69717402 0.87508695 0.08679497
 0.27826511 0.84900857 0.6920026  0.17914992]
Numba JIT: 0.0298s


## Célula 3 — Quando a lógica dificulta a vetorização

Agora o problema muda: a saída em cada posição depende do valor anterior (`res[i-1]`).

### O que isso significa
Esse tipo de dependência cria uma cadeia de cálculo em que cada passo usa o resultado do passo anterior.  
Isso reduz o potencial de paralelização direta e também dificulta a vetorização elegante com NumPy.

### O que a célula mostra
- **`logic_python`**: versão direta e didática, porém lenta.
- **`logic_numpy`**: tentativa de aproximar a ideia com operações vetoriais, mas sem reproduzir naturalmente toda a lógica dependente.
- **`logic_numba`**: mantém o loop dependente, mas compila para execução muito mais eficiente.

### O que observar
- Nem todo problema “gosta” de vetorização.
- Dependências entre iterações mudam completamente a estratégia de otimização.
- Em HPC, isso é central: **entender a estrutura da dependência antes de escolher a tecnologia**.

### Conceito didático
Este exemplo prepara o aluno para a ideia de que **GPUs e vetorização funcionam melhor em cargas homogêneas e independentes**, enquanto dependências sequenciais tendem a limitar o ganho.


In [13]:
import numpy as np
from numba import njit
import time

# 1. Função em Python Puro (Péssima performance)
def logic_python(arr):
    n = len(arr)
    res = np.zeros(n)
    for i in range(1, n):
        # Uma lógica condicional complexa que depende do vizinho
        if arr[i] > 0.5:
            res[i] = np.sqrt(arr[i]) + res[i-1]
        else:
            res[i] = arr[i]**2 - res[i-1]
    return res

# 2. Tentativa com NumPy (Vetorização parcial/difícil)
# Nota: Para lógicas onde o i depende do i-1, o NumPy costuma exigir 
# o uso de 'np.ufunc.accumulate', que é bem menos intuitivo.
def logic_numpy(arr):
    # O NumPy sofre aqui porque não consegue "olhar para trás" 
    # de forma nativa e eficiente sem loops em C.
    return np.cumsum(np.where(arr > 0.5, np.sqrt(arr), arr**2)) # Aproximação simplificada

# 3. Numba (O brilho: loop nativo e extremamente rápido)
@njit
def logic_numba(arr):
    n = len(arr)
    res = np.empty(n)
    res[0] = arr[0]
    for i in range(1, n):
        if arr[i] > 0.5:
            res[i] = np.sqrt(arr[i]) + res[i-1]
        else:
            res[i] = arr[i]**2 - res[i-1]
    return res

# --- Teste de Performance ---
n = 10_000_000
data = np.random.rand(n)

print(f"Processando {n} elementos...\n")

# Warm-up Numba (Compilação)
_ = logic_numba(data)

# Timing Numba
start = time.time()
res_numba = logic_numba(data)
print(f"Numba: {time.time()-start:.6f}s")

# Timing Python Puro (limitado a menos elementos para não travar)
n_small = 100_000
start = time.time()
_ = logic_python(data[:n_small])
print(f"Python Puro (escalado p/ {n}): ~{(time.time()-start)*(n/n_small):.2f}s")

# Timing NumPy (se tentarmos fazer o loop manual no NumPy sem vetorizar)
start = time.time()
_ = logic_python(data) # Rodando o loop manual no array numpy
print(f"NumPy com loop manual: {time.time()-start:.6f}s")

Processando 10000000 elementos...

Numba: 0.070709s
Python Puro (escalado p/ 10000000): ~4.68s
NumPy com loop manual: 4.282171s


## Célula 4 — CuPy: mesma operação vetorial, agora na GPU

Nesta célula, repetimos a multiplicação vetorial, mas usando **CuPy**, que opera sobre arrays alocados na GPU.

### Etapas executadas
1. Gerar os dados na CPU (`NumPy`).
2. Transferir os dados para a GPU (`cp.asarray`).
3. Executar a multiplicação vetorial na GPU.
4. Sincronizar a GPU para medir o tempo corretamente.
5. Trazer o resultado de volta para a CPU (`cp.asnumpy`).

### O que observar
- O tempo da multiplicação na GPU pode ser muito baixo.
- Porém, em problemas reais, também existe custo de **transferência CPU ↔ GPU**.
- Isso explica por que GPU não é automaticamente melhor para qualquer tarefa pequena.

### Conceito didático
Este é o primeiro contato concreto com a ideia de que desempenho em GPU depende de:
- volume de dados,
- paralelismo disponível,
- custo de movimentação de memória.


In [15]:
import cupy as cp
import numpy as np
import time

n = 10_000_000

# Dados na CPU (host)
a_cpu = np.random.rand(n)
b_cpu = np.random.rand(n)

# Transferir para GPU (device)
a_gpu = cp.asarray(a_cpu)
b_gpu = cp.asarray(b_cpu)

# Sincronizar antes de medir
cp.cuda.Stream.null.synchronize()

start = time.time()
c_gpu = a_gpu * b_gpu
cp.cuda.Stream.null.synchronize()
print(f"CuPy GPU: {time.time()-start:.5f}s")

# Trazer resultado de volta
c_cpu = cp.asnumpy(c_gpu)

CuPy GPU: 0.03699s


## Célula 5 — Uso não relacionado a IA: hashing em paralelo

Aqui a proposta é mostrar paralelismo fora do contexto de redes neurais ou tensores.

### O problema
Geramos muitos textos e calculamos o hash SHA-256 de cada um deles.

### O que a célula compara
- **Execução sequencial**: um hash por vez.
- **`multiprocessing.Pool`**: distribuição do trabalho entre múltiplos processos, aproveitando múltiplos núcleos da CPU.

### O que observar
- O problema é naturalmente divisível em tarefas independentes.
- Cada entrada pode ser processada sem depender das demais.
- Esse é um padrão clássico de paralelismo de dados/tarefas que aparece em segurança, ETL, processamento de arquivos e várias outras áreas além de IA.

### Conceito didático
A ideia aqui é reforçar que **usar paralelismo e acelerar código não é algo exclusivo de IA**.  
A GPU é importante no curso, mas o raciocínio de decomposição e medição de desempenho vale para vários domínios computacionais.


In [22]:
import hashlib
from multiprocessing import Pool
import time

def hash_data(data):
    return hashlib.sha256(
        data.encode()
    ).hexdigest()

# Gerar inputs
inputs = [f"dado_{i}" for i in range(10_000_000)]

# Sequencial (CPU single-thread)
start = time.time()
results_seq = [hash_data(d) for d in inputs]
print(f"Sequencial: {time.time()-start:.3f}s")

# Paralelo (CPU multi-core)
start = time.time()
with Pool() as pool:
    results_par = pool.map(hash_data, inputs)
print(f"Multiprocessing: {time.time()-start:.3f}s")

Sequencial: 6.311s
Multiprocessing: 6.231s
